**1. Why do we need VACUUM?**
Because Delta Lake is immutable and supports Time Travel, it never overwrites or deletes files during standard operations (UPDATE, DELETE, MERGE, or OPTIMIZE). Instead, it just creates new files and marks the old ones as "removed" in the transaction log.

Without VACUUM, your cloud storage (S3/ADLS/GCS) would grow forever, filled with "tombstoned" files from previous versions of your data.

**Retention Period (The Safety Buffer)**
The most important concept in VACUUM is the Retention Threshold.

Default: 168 hours (7 days).
Purpose: This buffer ensures that if someone is running a very long query that started 2 hours ago, VACUUM doesn't delete the files they are currently reading.
Time Travel Impact: Once you vacuum a table, you can no longer "Time Travel" back to a version older than the retention period.

Syntax: VACUUM table_name RETAIN 168 HOURS;

**The "Dry Run" (Best Practice)**
Before you actually delete data, you can see what would be deleted without actually performing the deletion. This is highly recommended for production tables:<br>
VACUUM table_name DRY RUN;


**Critical Limitations and Risks:**
- Irreversibility: Once a file is vacuumed, it is gone. You cannot "Undo" a vacuum.
- Breaking Time Travel: If you vacuum with a 0-hour retention, you lose all history immediately.
- The "Safety Check" Error: Databricks will prevent you from setting a retention period shorter than 7 days unless you manually override a safety setting:
# Use with extreme caution
spark.conf.set("spark.databricks.delta.retentionDurationCheck.enabled", "false")


